In [41]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import requests
import os
from dotenv import load_dotenv
from datetime import datetime
import folium

I'd like to implement an API that shows the current air quality reading for locations in Kentucky using PurpleAir.

In [ ]:
load_dotenv()

API_KEY = os.getenv("PURPLEAIR_READ_KEY")
BASE_URL = "https://api.purpleair.com/v1"

headers = {
    "X-API-Key": API_KEY
}

# --- Get a single sensor's data ---
sensor_index = 131075  # Replace with your target sensor's index

response = requests.get(
    f"{BASE_URL}/sensors/{sensor_index}",
    headers=headers,
    params={"fields": "name ,pm2.5, humidity, temperature"}
)

if response.status_code == 200:
    data = response.json()
    print(data)
else:
    print(f"Error {response.status_code}:", response.text)

{'api_version': 'V1.2.0-1.1.45', 'time_stamp': 1775417772, 'data_time_stamp': 1775417769, 'sensor': {'sensor_index': 131075, 'name': 'Mariners Bluff', 'humidity': 100, 'temperature': 83, 'pm2.5': 14.9}}


In [47]:
print(data['fields'])

['sensor_index', 'name', 'humidity', 'temperature', 'pm2.5']


In [ ]:
#Now responses for KY
response = requests.get(
    "https://api.purpleair.com/v1/sensors",
    headers=headers,
    params={
        "fields": "name, pm2.5, humidity, temperature, latitude, longitude",
        "location_type": 0,
        "nwlng": -89.57,
        "nwlat": 39.15,
        "selng": -81.96,
        "selat": 36.49,
    }
)

data = response.json()
print(f"Found {len(data['data'])} sensors")
print(data)

Found 119 sensors
{'api_version': 'V1.2.0-1.1.45', 'time_stamp': 1775418009, 'data_time_stamp': 1775417989, 'location_type': 0, 'max_age': 604800, 'firmware_default_version': '7.02', 'fields': ['sensor_index', 'name', 'latitude', 'longitude', 'humidity', 'temperature', 'pm2.5'], 'data': [[262243, 'TDEC APC - Kingsport - 262243', 36.538925, -82.52163, 37, 70, 0.7], [263785, 'EWV Whitman 1', 37.79283, -82.030815, 50, 61, 4.4], [263797, 'EWV Logan 2', 37.904606, -82.10735, 43, 62, 3.5], [263809, 'EWV Point Pleasant 3', 38.87221, -82.12882, 33, 66, 0.5], [273763, 'Boone Block', 39.08691, -84.50917, 36, 61, 0.3], [13031, 'Mount Vernon,Ind.', 37.94211, -87.89055, 27, 76, 0.8], [276356, 'Floyd Street Old Louisville', 38.23082, -85.75195, 26, 73, 1.9], [278523, 'EWV Five Mile 1', 37.742935, -82.14651, 47, 60, 2.8], [278531, 'EWV Logan 3', 37.881184, -82.081055, 45, 60, 3.3], [278566, 'EWV Point Pleasant 1', 38.91047, -82.11349, 44, 59, 1.3], [279406, 'Barbourville, KY / Knox County', 36.91824,

In [54]:
fields = data['fields']
sensors = data['data']

for sensor in sensors:
    sensor_dict = dict(zip(fields, sensor))
    print(f"Name: {sensor_dict['name']}")
    print(f"  PM2.5:       {sensor_dict['pm2.5']}")
    print(f"  Humidity:    {sensor_dict['humidity']}")
    print(f"  Temperature: {sensor_dict['temperature']}")
    print()

Name: TDEC APC - Kingsport - 262243
  PM2.5:       0.7
  Humidity:    37
  Temperature: 70

Name: EWV Whitman 1
  PM2.5:       4.4
  Humidity:    50
  Temperature: 61

Name: EWV Logan 2
  PM2.5:       3.5
  Humidity:    43
  Temperature: 62

Name: EWV Point Pleasant 3
  PM2.5:       0.5
  Humidity:    33
  Temperature: 66

Name: Boone Block
  PM2.5:       0.3
  Humidity:    36
  Temperature: 61

Name: Mount Vernon,Ind.
  PM2.5:       0.8
  Humidity:    27
  Temperature: 76

Name: Floyd Street Old Louisville
  PM2.5:       1.9
  Humidity:    26
  Temperature: 73

Name: EWV Five Mile 1
  PM2.5:       2.8
  Humidity:    47
  Temperature: 60

Name: EWV Logan 3
  PM2.5:       3.3
  Humidity:    45
  Temperature: 60

Name: EWV Point Pleasant 1
  PM2.5:       1.3
  Humidity:    44
  Temperature: 59

Name: Barbourville, KY / Knox County
  PM2.5:       1.1
  Humidity:    35
  Temperature: 66

Name: Cincinnati State
  PM2.5:       2.3
  Humidity:    42
  Temperature: 55

Name: Cincy Air Watch- Z

The API now gives all sensor readings available in Kentucky, about 119 so far until more people buy/register their PurpleAir sensors. With that, we can now make a map using Folium to show air quality readings across KY.

In [56]:
# Center map on Kentucky
m = folium.Map(location=[37.8, -85.8], zoom_start=7)

for sensor in sensors:
    sensor_dict = dict(zip(fields, sensor))
    
    folium.Marker(
        location=[sensor_dict['latitude'], sensor_dict['longitude']],
        popup=folium.Popup(
            f"<b>{sensor_dict['name']}</b><br>"
            f"PM2.5: {sensor_dict['pm2.5']}<br>"
            f"Humidity: {sensor_dict['humidity']}<br>"
            f"Temperature: {sensor_dict['temperature']}",
            max_width=200
        ),
        tooltip=sensor_dict['name']
    ).add_to(m)

m.save("kentucky_sensors_map.html")
print("Map saved.")

Map saved.


#####This API was made with the help of Claude AI in some parts to create a workable map and clarify how the API from PurpleAir was meant to be set up and used as it differs from a basic setup as we did in our API weather project that I used for comparison.